In [ ]:
!pip install -q langgraph

In [ ]:
import operator
import time

from IPython.display import Image
from enum import Enum
from google.colab import userdata
from langchain_core.runnables import Runnable, RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.types import Send
from pathlib import Path
from typing import Annotated, TypedDict

def display_graph(runnable: Runnable, output_png: Path) -> None:
    with output_png.open(mode="wb") as file:
        file.write(runnable.get_graph().draw_mermaid_png())

    display(Image(output_png, format="png"))

def explore_state_history(compiled_state_graph: CompiledStateGraph, config: RunnableConfig):
    state_history = list(compiled_state_graph.get_state_history(config))

    for snapshot in reversed(state_history):
        print(f"Step: {snapshot.metadata['step']}")
        print("Current state:")
        print(snapshot.values)
        print(f"Next: {snapshot.next}")
        print()

In [ ]:
class Team(Enum):
    BACK_END = "BE"
    FRONT_END = "FE"
    DEVOPS = "DOps"

class Task(TypedDict):
    team: Team
    title: str

class TaskAssignment(TypedDict):
    task_id: str
    task: Task

def merge_tasks(current: dict[str, Task], new: dict[str, Task]):
    return { **current, **new }

class SystemState(TypedDict):
    user_story: int
    tasks: Annotated[dict[str, Task], merge_tasks]
    finished: Annotated[set[str], operator.or_]

In [ ]:
def delegate(state: SystemState):
    tasks = state.get('tasks', {})
    finished = state.get('finished', set())

    free_slots = { Team.BACK_END: 1, Team.FRONT_END: 2, Team.DEVOPS: 1 }
    tasks_to_assign = { Team.BACK_END: [], Team.FRONT_END: [], Team.DEVOPS: [] }

    for task_id, task in tasks.items():
        if task_id in finished or free_slots[task['team']] == 0:
            continue

        free_slots[task['team']] -= 1
        tasks_to_assign[task['team']].append(task_id)

    result = []
    for task_id in tasks_to_assign[Team.BACK_END]:
        result.append(Send("back_end_team", { 'task_id': task_id, 'task': tasks[task_id] }))

    for task_id in tasks_to_assign[Team.FRONT_END]:
        result.append(Send("front_end_team", { 'task_id': task_id, 'task': tasks[task_id] }))

    # NOTE: Assign DevOps tasks only after back-end and front-end teams have finished.
    if not result:
        for task_id in tasks_to_assign[Team.DEVOPS]:
            result.append(Send("dev_ops_team", { 'task_id': task_id, 'task': tasks[task_id] }))

    if result:
        return result
    return END

In [ ]:
def supervisor(state: SystemState):
    user_story = state.get('user_story')

    return {
        "tasks": {
            f"{user_story}-be1": { 'team': Team.BACK_END, 'title': f"[{user_story}] Main back-end development" },
            f"{user_story}-be2": { 'team': Team.BACK_END, 'title': f"[{user_story}] Write integration tests" },
            f"{user_story}-be3": { 'team': Team.BACK_END, 'title': f"[{user_story}] PR Corrections" },
            f"{user_story}-fe1": { 'team': Team.FRONT_END, 'title': f"[{user_story}] Main front-end development" },
            f"{user_story}-fe2": { 'team': Team.FRONT_END, 'title': f"[{user_story}] Improve UI and UX" },
            f"{user_story}-dops": { 'team': Team.DEVOPS, 'title': f"[{user_story}] Deploy to staging" },
        }
    }

def create_worker_node(expertise: str, wait_seconds: int):
    def worker(task_assignment: TaskAssignment):
        print(f"I am an expert in {expertise} and I will start working over a new task:\n{task_assignment['task']['title']}")

        time.sleep(wait_seconds)
        print(f"Finished task \"{task_assignment}\".")

        return { "finished": {task_assignment['task_id']} }

    return worker

In [ ]:
in_memory_checkpointer = InMemorySaver()

In [ ]:
builder = StateGraph(SystemState)
builder.add_node('supervisor', supervisor)
builder.add_node('front_end_team', create_worker_node("front-end development", 1))
builder.add_node('back_end_team', create_worker_node("back-end development", 3))
builder.add_node('dev_ops_team', create_worker_node("DevOps", 5))

builder.add_edge(START, 'supervisor')
builder.add_conditional_edges('supervisor', delegate, ['front_end_team', 'back_end_team', 'dev_ops_team', END])
builder.add_edge('front_end_team', 'supervisor')
builder.add_edge('back_end_team', 'supervisor')
builder.add_edge('dev_ops_team', 'supervisor')

graph = builder.compile(checkpointer=in_memory_checkpointer)

In [ ]:
display_graph(graph, Path("/content/graph.png"))

In [ ]:
thread1_config = { "configurable": { "thread_id": "thread_1" } }
final_state = graph.invoke(input={ "user_story": 2154 }, config=thread1_config)

In [ ]:
final_state

In [ ]:
explore_state_history(graph, thread1_config)